# PROJECT 1 · STEP 0 — Compression Molding 공정 이해

**분석 범위:** chip-first reconstituted wafer의 wafer-level compression molding project scenario  
**현재 단계의 목적:** 데이터를 보기 전에 공정의 물리적 경계, 입력, 출력, 설비·소재·측정 요인을 정의한다.  
**주의:** 아래 recipe와 설비 항목은 분석 설계용 변수 후보이며 특정 회사의 생산 조건이나 표준값이 아니다.

## 1. 공정 목적과 패키지 구조

Compression molding은 carrier 위에 배치된 die를 epoxy molding compound(EMC)로 봉지하여 기계적 충격, 습기와 오염으로부터 보호하고 후속 redistribution layer(RDL)를 형성할 수 있는 molded wafer를 만든다. Transfer molding보다 EMC의 유동 거리가 짧고 낮은 전단 조건을 구성할 수 있어 얇고 넓은 wafer-level 구조에 사용된다.

본 프로젝트의 관찰 대상은 두 가지다.

- **Edge void:** 최종 검사에서 wafer 외곽부에 집중되는 미충전 또는 포획 기체 영역
- **Chip offset:** molding 전 기준 위치 대비 molding/냉각/탈착 후 die 중심의 이동 벡터 `(dx, dy)`

두 현상은 독립적일 수도 있지만, 비대칭 EMC flow front, pressure gradient, vacuum/vent imbalance가 동시에 존재하면 같은 방향성을 보일 수 있다. 반면 chip offset에는 열팽창·수축과 carrier/tape 거동도 포함되므로 flow drag만으로 단정하지 않는다.

## 2. 장비 구조 — 기능 단위

| 장비 모듈 | 기능 | 관측 가능한 신호 | 관련 failure signature |
|---|---|---|---|
| Upper/lower mold & platen | cavity 형성, 가열, 압축 하중 전달 | zone temperature, platen position, force/pressure | zone별 온도 편차, 비대칭 충전 |
| Vacuum chamber/pump/line | 충전 전·중 cavity 내 기체 제거 | absolute pressure, evacuation time, leak-up rate | edge void, shot 간 불안정 |
| Vent path | 잔류 기체와 volatile 배출 | 직접 센서가 없으면 cleaning count/차압 proxy | 특정 방향·edge 반복 void |
| EMC loading/dispense | EMC charge의 질량·형상·위치 제어 | charge mass, placement offset, material lot | flow imbalance, short fill |
| Release film & tension unit | mold 오염 방지, demold 보조 | left/right tension, roll lot, usage count | 방향성 offset, wrinkle/imprint |
| Carrier/chuck | die array 지지와 평탄도 유지 | chuck ID, flatness, vacuum zone | 국부 gap, 반복 위치 패턴 |
| Motion/press control | closing speed, compression profile, cure hold | position-time, pressure-time, alarm | drag 증가, cure 불균일 |
| Trace/data acquisition | recipe와 센서 waveform 저장 | timestamp, sampling rate, calibration status | drift 또는 순간 excursion 검출 누락 |

## 3. 공정 Sequence와 물리 현상

1. **Carrier 준비 및 die placement** — release/adhesive tape 위에 die array를 배치한다. 초기 좌표와 접착 상태가 chip offset의 기준이다.
2. **EMC 준비·투입** — EMC lot, 보관 시간, 노출 이력, charge 질량과 배치가 초기 점도와 flow symmetry를 좌우한다.
3. **Mold loading 및 vacuum evacuation** — cavity의 공기를 제거한다. evacuation 도달압뿐 아니라 도달 시간, leak-up, waveform 안정성이 중요하다.
4. **Preheat/softening** — EMC 점도가 낮아져 흐를 수 있게 된다. 온도가 너무 낮으면 높은 유동 저항, 너무 높거나 지연이 길면 조기 gel 위험이 있다.
5. **Compression/filling** — platen 이동과 압력으로 EMC가 die 사이와 edge로 흐른다. 점성 drag와 pressure gradient가 die에 횡력을 가할 수 있다.
6. **Cure/hold** — 수지가 경화한다. 시간·온도 불균일은 cure state, 수축, 잔류응력의 공간 차이를 만든다.
7. **Mold open/demold** — release film과 mold에서 molded wafer를 분리한다. sticking이나 film tension 비대칭은 추가 변형 가능성이 있다.
8. **Post-mold cure/cooling/debond** — 열수축, CTE mismatch, carrier 제거에 따른 warpage와 apparent die movement가 더해질 수 있다.
9. **Inspection** — SAM/X-ray/optical metrology로 void, die 좌표, warpage를 측정하고 공정 trace와 join한다.

**중요한 분리:** molding 직후와 debond 후 좌표를 모두 확보해야 flow-induced shift와 열/warpage-induced apparent shift를 구분할 수 있다.

## 4. Input과 Parameter 분류

### Recipe / Method parameter

- vacuum setpoint와 evacuation time
- preheat temperature/time
- compression speed, pressure profile, hold time
- mold/cure temperature와 cure time
- EMC charge mass와 placement 위치

### Equipment parameter

- actual vacuum trace, leak-up rate, pump-down slope
- platen parallelism, zone별 actual temperature, pressure uniformity
- vent condition/cleaning age, chuck ID/flatness
- release film left/right tension, mold cycle count

### Material parameter

- EMC lot, storage time, floor-life exposure, moisture
- viscosity-vs-temperature curve, gel time, filler content
- release film lot/thickness, adhesive tape lot/adhesion
- die surface contamination와 carrier 상태

### Geometry / incoming parameter

- die size, thickness, gap, radial position
- initial die placement `(x0, y0, theta0)`
- cavity thickness, EMC initial charge geometry
- edge distance와 local pattern density

## 5. Output/CTQ와 대표 Failure Mode

| Output 후보 | 정의 | 대표 failure mode | 잠재 영향 |
|---|---|---|---|
| Edge void ratio | edge band 검사 면적 중 void 면적 비율 | air pocket, incomplete fill | 신뢰성/외관/후속 공정 risk |
| Void size/count | 개별 void 면적 또는 등가직경과 개수 | isolated/connected void | 국부 취약부 |
| Chip offset magnitude | `sqrt(dx²+dy²)` | translation | RDL alignment margin 감소 |
| Chip offset direction | `atan2(dy,dx)` | coherent radial/tangential shift | 비대칭 flow/equipment 진단 |
| Rotation | molding 전후 die angle 차이 | die rotation | overlay 불량 |
| Warpage | 기준 평면 대비 out-of-plane 변위 | bow/twist | handling 및 후속 공정 문제 |
| Cure/adhesion proxy | cure state 또는 계면 검사 | delamination, crack | 장기 신뢰성 risk |
| Cycle time | load부터 unload까지 시간 | takt 증가 | 생산성 저하 |

STEP 0에서는 규격 숫자를 정의하지 않는다. Target/USL/현재 수준/개선 목표는 STEP 1에서 모두 **Engineering Target**으로 명시한다.

## 6. 주요 Sensor와 실제 수집 가능 데이터

| 데이터 grain | 필수 key | 예시 변수 | 진단 목적 |
|---|---|---|---|
| Die/defect | lot, wafer, die_id, x, y | edge_distance, void area, dx, dy, contamination | 공간 pattern과 방향성 |
| Shot/wafer | equipment, chamber, timestamp | recipe, cycle time, alarms | lot/설비/time dependency |
| Waveform | shot_id, elapsed time | vacuum, pressure, platen position, zone temperature | 순간 excursion과 sequence 비교 |
| Material | EMC lot, release film lot | storage/floor time, incoming property | material-lot dependency |
| Maintenance | equipment/chamber/part | vent cleaning, PM, film/chuck change | change point와 회복 확인 |
| Metrology | tool, recipe, operator/time | SAM/X-ray threshold, coordinate residual, repeat scan | 측정 산포 분리 |

평균값만 저장하면 짧은 vacuum loss나 temperature overshoot를 놓친다. 따라서 shot summary와 원 waveform을 함께 보존하고, 공정 clock를 동기화해야 한다.

## 7. 4M + Measurement 분류

| 범주 | 원인 후보 | 예상 data signature | 우선 확인 데이터 |
|---|---|---|---|
| Machine | vacuum leak, vent 막힘, zone heater 편차, platen parallelism, film tension imbalance | 특정 chamber/방향 반복, PM 후 step change | trace, chamber, PM, left/right tension |
| Material | EMC viscosity/gel time, 보관·노출, moisture, film/tape lot | EMC lot 교체와 동행, 여러 장비에서 재현 | material genealogy, storage/floor time |
| Method | evacuation/closing/cure profile, charge 위치 | recipe revision 이후 shift, setting-dose response | recipe version, actual trace |
| Man | loading 방향, cleaning/교체 편차, 취급 오염 | operator/shift 의존성; 표준화 후 감소 | operator, shift, checklist |
| Measurement | SAM threshold, 좌표 정합, scan resolution, tool drift | 재측정 결과 불일치, tool/recipe 의존 | repeat scan, reference sample, calibration |
| Environment | 습도, 대기시간, utility 변동 | 계절/시간대 또는 queue time 의존 | RH, ambient T, utility, queue time |

### 공정기인·설비기인·소재기인 구분 논리

- **설비기인:** 동일 소재/recipe에서도 특정 equipment 또는 chamber에서 재현되고 PM·부품 교체 후 회복한다.
- **소재기인:** 동일 EMC lot이 여러 설비에서 같은 방향의 악화를 만들며 storage/floor time 또는 incoming property와 연결된다.
- **공정/Method 기인:** actual parameter dose와 CTQ 사이 재현 가능한 반응이 있고 recipe 변경/DOE에서 효과가 확인된다.
- **측정기인:** 동일 시료 재측정/도구 변경으로 판정이 바뀌며 physical trace나 인접 CTQ와 일치하지 않는다.

한 요인만으로 단정하지 않고 chamber×material lot, recipe×material, radial position×flow direction 같은 interaction을 확인한다.

## 8. STEP 0 Engineering Gate

### 확인된 출발점

- Void는 단순 defect count가 아니라 edge distance, 면적, 방향을 가진 공간 사건으로 다룬다.
- Chip offset은 magnitude만 보지 않고 vector field로 분석한다.
- Vacuum은 setpoint가 아니라 waveform, 도달 시간, leak-up까지 본다.
- EMC flow drag와 thermal/warpage contribution을 분리하기 위해 공정 단계별 die 좌표가 필요하다.
- 개선 시 warpage와 cycle time을 secondary CTQ로 추적한다.

### STEP 1로 넘길 질문

1. 대표 Y를 edge void ratio로 두고 chip offset을 동시 CTQ로 둘 것인가?
2. Edge band를 wafer 반경의 몇 %로 정의할 것인가?
3. Void 판정은 area ratio, max void size, defect die rate 중 무엇을 주지표로 할 것인가?
4. Target/USL/현재 수준/개선 목표의 project scenario 값을 어떻게 설정할 것인가?
5. Side effect의 우선순위를 warpage와 cycle time 중 어디에 둘 것인가?

## 9. 공개 문헌 근거와 한계

1. Yeon et al. (2016), *Compensation Method for Die Shift Caused by Flow Drag Force in Wafer-Level Molding Process*. Die shift를 EMC flow drag, 열팽창/수축, warpage 영향으로 분해할 필요를 뒷받침한다. https://doi.org/10.3390/mi7060095
2. Guo & Young (2015), *Vacuum effect on the void formation of the molded underfill process in flip chip packaging*. 진공 품질과 포획 air pocket의 관계를 실험적으로 다룬다. https://doi.org/10.1016/j.microrel.2014.12.001
3. Hsu et al. (2025), *Compression Molding Flow Behavior and Void Optimization of an Integrated Circuit Package with Shielding-Metal-Frame*. 비대칭 구조, vent/vacuum을 포함한 flow behavior와 void 위치의 관계를 다룬다. https://doi.org/10.3390/polym17101301

문헌은 메커니즘과 변수 선택의 근거로만 사용한다. 이후 생성할 수치 범위와 인과계수는 실제 Fab recipe가 아니라 명시적인 project scenario이며, synthetic data 결과를 생산 성능으로 표현하지 않는다.